# IK end-effector control

This notebook replaces the terminal UI from `../../WBC/ik_pose_cli_v3.py` with a Jupyter widget panel. It sends small Cartesian end-effector increments through `ArmSdk.ik_move_EE()` and clamps joint changes per command with `max_dq`.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import `ArmSdk`, widgets, and numeric helpers.


In [ ]:
import json
import time
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from arm_sdk import ArmSdk


Create the IK controller and sync it to the current measured upper-body state.


In [ ]:
ik = ArmSdk(iface=IFACE, domain_id=DOMAIN_ID)
ik.resync()
print("ArmSdk IK controller synced to current state.")


Helpers for EE pose display and single-increment commands.


In [ ]:
DOF_INDEX = {"x": 0, "y": 1, "z": 2, "roll": 3, "pitch": 4, "yaw": 5}


def pose_summary(arm):
    T = ik.get_ee_pose(arm)
    return {
        "arm": arm,
        "position_xyz_m": [round(float(v), 4) for v in T[:3, 3]],
        "rotation_matrix": [[round(float(v), 4) for v in row] for row in T[:3, :3]],
    }


def ramped_ik_move_EE(
    pose_increment,
    *,
    arm="right",
    position_only=False,
    selected_axis=None,
    max_dq=0.04,
    cart_speed_m_s=0.04,
    rot_speed_rad_s=0.25,
    rate_hz=20.0,
):
    """Apply a Cartesian increment as small timed IK steps.

    `ArmSdk.ik_move_EE()` solves and publishes one IK target. Splitting the
    pose increment here gives the same safety intent as ik_pose_cli_v3: one
    request becomes a ramped sequence rather than a single large joint jump.
    """
    inc = np.asarray(pose_increment, dtype=np.float64)
    if inc.shape != (6,):
        raise ValueError(f"pose_increment must have 6 elements, got {inc.shape}")
    rate = max(1.0, float(rate_hz))
    dt = 1.0 / rate
    trans_steps = int(np.ceil(float(np.max(np.abs(inc[:3]))) / max(1e-4, float(cart_speed_m_s) * dt)))
    rot_steps = int(np.ceil(float(np.max(np.abs(inc[3:]))) / max(1e-4, float(rot_speed_rad_s) * dt)))
    steps = max(1, trans_steps, rot_steps)
    step_inc = inc / float(steps)
    infos = []
    for step_idx in range(steps):
        info = ik.ik_move_EE(
            step_inc,
            arm=arm,
            position_only=bool(position_only),
            selected_axis=selected_axis,
            max_dq=float(max_dq),
        )
        infos.append(info)
        if step_idx + 1 < steps:
            time.sleep(dt)
    return {"steps": steps, "step_increment": [float(v) for v in step_inc], "last": infos[-1] if infos else None}


def apply_increment(
    arm,
    dof,
    signed_step,
    position_only=False,
    max_dq=0.04,
    cart_speed_m_s=0.04,
    rot_speed_rad_s=0.25,
    rate_hz=20.0,
):
    inc = np.zeros(6, dtype=np.float64)
    inc[DOF_INDEX[dof]] = float(signed_step)
    selected_axis = DOF_INDEX[dof] if dof in {"x", "y", "z"} else None
    return ramped_ik_move_EE(
        inc,
        arm=arm,
        position_only=bool(position_only),
        selected_axis=selected_axis,
        max_dq=float(max_dq),
        cart_speed_m_s=float(cart_speed_m_s),
        rot_speed_rad_s=float(rot_speed_rad_s),
        rate_hz=float(rate_hz),
    )


Run the panel. Use small translation and rotation steps, and resync after physical contact or manual repositioning.


In [ ]:
arm = widgets.ToggleButtons(options=["left", "right", "both"], value="right", description="Arm")
dof = widgets.Dropdown(options=list(DOF_INDEX), value="x", description="DOF")
translation_step = widgets.FloatSlider(value=0.02, min=0.002, max=0.08, step=0.002, description="m step")
rotation_step = widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, description="rad step")
max_dq = widgets.FloatSlider(value=0.04, min=0.01, max=0.12, step=0.005, description="max dq")
cart_speed = widgets.FloatSlider(value=0.04, min=0.005, max=0.12, step=0.005, description="m/s")
rot_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description="rad/s")
ramp_rate = widgets.FloatSlider(value=20.0, min=5.0, max=50.0, step=1.0, description="Hz")
position_only = widgets.Checkbox(value=False, description="free orientation for xyz")
minus = widgets.Button(description="- Step")
plus = widgets.Button(description="+ Step", button_style="success")
resync = widgets.Button(description="Resync", button_style="info")
status = widgets.Textarea(layout=widgets.Layout(width="100%", height="260px"), disabled=True)


def current_step():
    return translation_step.value if dof.value in {"x", "y", "z"} else rotation_step.value


def refresh(extra=None):
    arms = ["left", "right"] if arm.value == "both" else [arm.value]
    payload = {"status": extra, "poses": [pose_summary(a) for a in arms]}
    status.value = json.dumps(payload, indent=2)


def move(sign):
    try:
        info = apply_increment(arm.value, dof.value, sign * current_step(), position_only.value, max_dq.value, cart_speed.value, rot_speed.value, ramp_rate.value)
        refresh(info)
    except Exception as exc:
        refresh({"error": str(exc)})

minus.on_click(lambda _: move(-1.0))
plus.on_click(lambda _: move(+1.0))
resync.on_click(lambda _: (ik.resync(), refresh("resynced")))
refresh("ready")
display(widgets.VBox([widgets.HBox([arm, dof, position_only]), widgets.HBox([translation_step, rotation_step, max_dq]), widgets.HBox([cart_speed, rot_speed, ramp_rate]), widgets.HBox([minus, plus, resync]), status]))
